# Pandas solutions: pandas

Worked solutions to the exercises. Other correct answers are possible; what matters is that the result is right and the code is readable.

## Exercise 1

Load `patients.csv`. Print the number of rows and columns, the column types, and the first five rows.

In [1]:
import pandas as pd
import numpy as np
RAW = "../../datasets/raw/"
patients = pd.read_csv(RAW + "patients.csv", parse_dates=["enrolled"])
print(patients.shape)
print(patients.dtypes)
patients.head()

(300, 9)
patient_id               str
site                     str
sex                      str
age                    int64
weight_kg            float64
height_cm            float64
smoker                   str
group                    str
enrolled      datetime64[us]
dtype: object


,patient_id,site,sex,age,weight_kg,height_cm,smoker,group,enrolled
0,P0001,Site A,M,41,66.4,156.3,No,Treatment,2025-03-31
1,P0002,Site B,M,83,69.5,168.0,No,Control,2025-08-19
2,P0003,Site B,F,36,71.4,172.2,No,Control,2025-08-11
3,P0004,Site A,F,24,66.0,158.9,Yes,Control,2025-06-24
4,P0005,Site A,M,77,80.5,163.7,No,Control,2025-08-05


## Exercise 2

Create a DataFrame by hand with columns `sample`, `type` and `value` for four samples, then add a column `value_x10`.

In [2]:
d = pd.DataFrame({"sample": ["S1", "S2", "S3", "S4"], "type": ["Blood", "Urine", "Blood", "Saliva"],
                  "value": [5.1, 2.3, 6.8, 1.9]})
d["value_x10"] = d["value"] * 10
d

,sample,type,value,value_x10
0,S1,Blood,5.1,51.0
1,S2,Urine,2.3,23.0
2,S3,Blood,6.8,68.0
3,S4,Saliva,1.9,19.0


## Exercise 3

Use `describe` on the patients' numeric columns. Which column has missing values, and how many?

In [3]:
print(patients.describe().round(1))
print(patients.isna().sum()[patients.isna().sum() > 0])

         age  weight_kg  height_cm             enrolled
count  300.0      290.0      300.0                  300
mean    51.4       70.0      165.1  2025-06-30 02:33:36
min     18.0       38.0      140.1  2025-01-02 00:00:00
25%     33.0       61.4      158.9  2025-04-03 12:00:00
50%     51.0       69.6      165.3  2025-06-23 12:00:00
75%     70.0       78.6      171.9  2025-09-27 06:00:00
max     85.0      101.6      190.4  2025-12-31 00:00:00
std     20.1       12.7        9.4                  NaN
weight_kg    10
dtype: int64


weight_kg has 10 missing values (count 290 out of 300).

## Exercise 4

How many patients are there in each site, and what percentage are smokers overall?

In [4]:
print(patients["site"].value_counts())
print(f"{(patients['smoker'] == 'Yes').mean():.1%}")

site
Site A    111
Site B     87
Site C     54
Site D     48
Name: count, dtype: int64
23.0%


## Exercise 5

Select female patients over 60 from Site B, showing only patient_id, age and group, sorted by age (oldest first).

In [5]:
(patients.query("sex == 'F' and age > 60 and site == 'Site B'")
 [["patient_id", "age", "group"]].sort_values("age", ascending=False))

,patient_id,age,group
88,P0089,84,Treatment
71,P0072,83,Control
243,P0244,82,Treatment
86,P0087,80,Control
221,P0222,78,Treatment
264,P0265,77,Treatment
56,P0057,77,Control
275,P0276,76,Treatment
21,P0022,75,Treatment
228,P0229,74,Control


## Exercise 6

Add a BMI column (height is in cm) and a column `bmi_class` with Underweight / Normal / Overweight / Obese. How many patients fall in each class?

In [6]:
patients["bmi"] = patients["weight_kg"] / (patients["height_cm"] / 100) ** 2
patients["bmi_class"] = pd.cut(patients["bmi"], [0, 18.5, 25, 30, 100],
                               labels=["Underweight", "Normal", "Overweight", "Obese"], right=False)
patients["bmi_class"].value_counts(dropna=False)

bmi_class
Normal         116
Overweight      79
Obese           68
Underweight     27
NaN             10
Name: count, dtype: int64

`right=False` makes each bin include its left edge, so BMI 25.0 is Overweight, as in the standard definition.

## Exercise 7

Fill missing weights with the median weight of the same sex, and confirm no weights are missing afterwards.

In [7]:
filled = patients.copy()
filled["weight_kg"] = filled["weight_kg"].fillna(filled.groupby("sex")["weight_kg"].transform("median"))
print(filled["weight_kg"].isna().sum())

0


## Exercise 8

Load `lab_results.csv`. Compute the mean glucose, haemoglobin and systolic BP at each visit number.

In [8]:
labs = pd.read_csv(RAW + "lab_results.csv", parse_dates=["visit_date"])
labs.groupby("visit")[["glucose_mmol", "haemoglobin_g_dl", "systolic_bp"]].mean().round(2)

,glucose_mmol,haemoglobin_g_dl,systolic_bp
visit,,,
0,6.18,13.19,127.63
1,6.08,13.37,127.52
2,5.92,13.14,126.36
3,5.84,12.99,128.75


## Exercise 9

Merge the lab results with the patients' group and compute, per group, the number of visits, the mean glucose and the proportion of visits with glucose of 7.0 or more.

In [9]:
m = labs.merge(patients[["patient_id", "group"]], on="patient_id", how="left", validate="many_to_one")
m.groupby("group").agg(visits=("visit", "size"), mean_glucose=("glucose_mmol", "mean"),
                       prop_high=("glucose_mmol", lambda g: (g >= 7).mean())).round(3)

,visits,mean_glucose,prop_high
group,,,
Control,458,6.158,0.245
Treatment,436,5.941,0.161


## Exercise 10

Reshape the systolic BP into wide format (one row per patient, one column per visit), then back to long format.

In [10]:
wide = labs.pivot(index="patient_id", columns="visit", values="systolic_bp")
print(wide.head())
long = wide.reset_index().melt(id_vars="patient_id", value_name="systolic_bp").dropna()
long.head()

visit           0      1      2      3
patient_id                            
P0001       119.0  101.0    NaN    NaN
P0002       105.0  103.0  135.0    NaN
P0003       140.0  121.0    NaN    NaN
P0004       105.0  106.0  105.0    NaN
P0005       114.0  126.0  134.0  130.0


,patient_id,visit,systolic_bp
0,P0001,0,119.0
1,P0002,0,105.0
2,P0003,0,140.0
3,P0004,0,105.0
4,P0005,0,114.0


## Exercise 11

From `daily_sales.csv`, compute total revenue (units × price) per shop per year, as a table with shops as rows and years as columns.

In [11]:
sales = pd.read_csv(RAW + "daily_sales.csv", parse_dates=["date"])
sales["revenue"] = sales["units"] * sales["price"]
sales.pivot_table(index="shop", columns=sales["date"].dt.year, values="revenue", aggfunc="sum").round(0)

date,2024,2025
shop,,
Central,436729.0,441382.0
North,204077.0,202265.0
South,151792.0,150946.0


## Exercise 12

In the sales data, compare the mean units sold on promotion days and other days, for each shop.

In [12]:
sales.groupby(["shop", "promo"])["units"].mean().unstack().round(1)

promo,0,1
shop,,
Central,262.3,387.9
North,120.4,194.3
South,90.3,138.8


## Exercise 13

Load the Seaborn `tips` dataset with `pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")`. Add a `tip_pct` column (tip as a percentage of total_bill) and find the mean tip percentage by day and time.

In [13]:
tips = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
tips["tip_pct"] = tips["tip"] / tips["total_bill"] * 100
tips.pivot_table(index="day", columns="time", values="tip_pct", aggfunc="mean", observed=True).round(1)

time,Dinner,Lunch
day,,
Fri,15.9,18.9
Sat,15.3,NaN
Sun,16.7,NaN
Thur,16.0,16.1


## Exercise 14

Load the Seaborn `penguins` dataset in the same way (`penguins.csv`). How many rows have any missing value? Drop them, then give the mean body mass per species and sex.

In [14]:
peng = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
print(peng.isna().any(axis=1).sum(), "rows with a missing value")
peng = peng.dropna()
peng.groupby(["species", "sex"])["body_mass_g"].mean().unstack().round(0)

11 rows with a missing value


sex,FEMALE,MALE
species,,
Adelie,3369.0,4043.0
Chinstrap,3527.0,3939.0
Gentoo,4680.0,5485.0
